SETUP

In [ ]:
# =========================
# Setup: Install dependencies (run once)
# =========================

!pip install -q numpy scipy scikit-learn transformers sentence-transformers beautifulsoup4 lxml google-genai python-dotenv

In [ ]:
# =========================
# Load Gemini API key from BUFN403/.env (or set GEMINI_API_KEY in Colab secrets)
# =========================

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    for _base in [Path.cwd(), *Path.cwd().parents]:
        _env = _base / ".env"
        if _env.is_file():
            load_dotenv(_env, override=False)
            break
except ImportError:
    pass

if not os.getenv("GEMINI_API_KEY"):
    raise EnvironmentError(
        "Set GEMINI_API_KEY: add BUFN403/.env with GEMINI_API_KEY=... "
        "or set the variable in your environment / Colab secrets."
    )

# =========================
# Check bank folders (local)
# =========================

!ls banks

ValueError: Mountpoint must not already contain files

IMPORTS

In [ ]:
from pathlib import Path
import os
import re
import json
import time
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from google import genai

Cell 2 — Configuration

Sets the main runtime settings for the notebook: where the bank folders live, where outputs should be written, which banks are available, which tickers to process in this run, and which embedding / Gemini models to use.

In [ ]:
# -------------------------
# CONFIG
# -------------------------


ROOT_DIR = Path("banks")
OUTPUT_DIR = Path("private_credit_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BANKS = {
    "PNFP": {"name": "Pinnacle Financial Partners", "idrssd": 2925666},
    "CFR":  {"name": "Cullen/Frost Bankers", "idrssd": 682563},
    "BOKF": {"name": "BOK Financial Corp.", "idrssd": 339858},
    "FNB":  {"name": "F.N.B. Corporation", "idrssd": 379920},
    "SOFI": {"name": "SoFi Technologies", "idrssd": 962966},
    "ASB":  {"name": "Associated Banc-Corp", "idrssd": 917742},
    "SF":   {"name": "Stifel Financial Corp.", "idrssd": 3076220},
    "PB":   {"name": "Prosperity Bancshares", "idrssd": 664756},
    "BKU":  {"name": "BankUnited, Inc.", "idrssd": 3938186},
}

TICKERS_TO_RUN = ["PNFP", "CFR", "BOKF", "FNB", "SOFI", "ASB", "SF", "PB", "BKU"]

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEMINI_MODEL_NAME = "gemini-2.5-flash"


NameError: name 'Path' is not defined

Cell 3 — Query phrases and keyword groups

Defines the private-credit concepts the pipeline is looking for. The query phrases are used for semantic similarity ranking, while the keyword groups add a small boost for exact matches like “private equity,” “CLO,” or “asset-based lending.”

In [ ]:
QUERY_PHRASES = [
    "sponsor-backed lending to private equity owned companies",
    "financial sponsor lending and middle-market sponsor finance",
    "direct lending private credit private debt platform",
    "lending to finance companies asset managers hedge funds private funds",
    "collateralized loan obligations CLO leveraged loan securitization",
    "structured finance structured credit specialty finance asset-based lending warehouse financing",
    "institutional banking middle-market lending sponsor-backed borrowers",
]

KEYWORD_GROUPS = {
    "sponsor_pe": [
        "sponsor-backed", "financial sponsor", "private equity",
        "private-equity", "sponsor finance", "middle-market sponsor"
    ],
    "direct_lending": [
        "direct lending", "private credit", "private debt",
        "middle-market direct lending", "credit platform"
    ],
    "financial_institutions": [
        "finance companies", "asset managers", "hedge funds",
        "private funds", "specialty finance", "institutional clients"
    ],
    "clo": [
        "clo", "collateralized loan obligations",
        "loan securitization", "securitization"
    ],
    "structured_finance": [
        "structured finance", "structured credit",
        "asset-based lending", "warehouse financing",
        "specialty finance"
    ],
}

Cell 4 — Gemini prompt template

Creates the instruction template sent to Gemini. This forces the model to return a consistent bank report structure with the specific sections you want and a final 1–10 involvement score.

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a financial analyst focused on bank SEC filings and private credit markets.

You will receive curated excerpts from one bank's two most recent 10-K filings.
Your task is to write a concise qualitative report about the bank's involvement in private credit.

Use ONLY the provided text.
Do not invent facts.
If evidence is weak or absent, say so explicitly.

Write the report in markdown with exactly these sections:

# {bank_name}

## Overview
2-4 sentences on the bank's likely involvement in private credit.

## Sponsor-Backed / Private Equity Lending
- Bullet points with evidence or note that no clear evidence was found.

## Direct Lending / Private Credit Platforms
- Bullet points with evidence or note that no clear evidence was found.

## Lending to Financial Institutions
- Bullet points with evidence or note that no clear evidence was found.

## CLO Activity
- Bullet points with evidence or note that no clear evidence was found.

## Structured / Specialty Finance
- Bullet points with evidence or note that no clear evidence was found.

## Final Assessment
1 short paragraph summarizing the bank's position.

## Score
Score: X/10

Scoring guide:
1-2 = no meaningful evidence
3-4 = low involvement
5-6 = moderate involvement
7-8 = strong involvement
9-10 = very heavy involvement

Base the score only on the supplied text.
"""

Cell 5 — Filing section patterns

Defines regex patterns used to detect major 10-K sections, especially Item 1 and Item 7. These sections are prioritized because they are the most likely to describe the bank’s business model and lending activities.

In [ ]:
SECTION_PATTERNS = {
    "item1": [
        r"\bitem\s+1[\.\-:\s]+business\b",
        r"\bitem\s+1\b",
    ],
    "item1a": [
        r"\bitem\s+1a[\.\-:\s]+risk factors\b",
        r"\bitem\s+1a\b",
    ],
    "item7": [
        r"\bitem\s+7[\.\-:\s]+management[’'`s]{0,2}\s+discussion",
        r"\bitem\s+7\b",
    ],
    "item7a": [
        r"\bitem\s+7a\b",
    ],
}

Cell 6 — Basic text helper functions

Contains utility functions for cleaning raw filing text, converting HTML to plain text, and saving outputs. This is the first stage of turning messy SEC filing files into text the pipeline can work with.

In [ ]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def save_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def html_to_text(path: Path) -> str:
    html = path.read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    return normalize_whitespace(text)

def txt_to_text(path: Path) -> str:
    return normalize_whitespace(path.read_text(encoding="utf-8", errors="ignore"))

Cell 7 — Filing reading and selection

Handles locating the two most recent 10-K folders for a bank and reading the filing contents from either primary-document.html or full-submission.txt. This is how the notebook chooses which source documents to analyze.

In [ ]:
def read_filing_text(filing_dir: Path) -> str:
    html_path = filing_dir / "primary-document.html"
    txt_path = filing_dir / "full-submission.txt"
    if html_path.exists():
        return html_to_text(html_path)
    if txt_path.exists():
        return txt_to_text(txt_path)
    raise FileNotFoundError(f"No filing file found in {filing_dir}")

def find_latest_two_10k_dirs(bank_dir: Path):
    tenk_dir = bank_dir / "10-K"
    if not tenk_dir.exists():
        return []
    dirs = [p for p in tenk_dir.iterdir() if p.is_dir()]
    dirs.sort(key=lambda p: p.name)
    return dirs[-2:]

Cell 8 — Section extraction

Tries to carve the filing into high-value regions, mainly Item 1 and Item 7. If exact section headings are not found, it falls back to approximate slices of the document so the pipeline can still continue.

In [ ]:
def first_match_span(text: str, patterns):
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.span()
    return None

def extract_sections(text: str):
    lower = text.lower()
    item1_span = first_match_span(lower, SECTION_PATTERNS["item1"])
    item1a_span = first_match_span(lower, SECTION_PATTERNS["item1a"])
    item7_span = first_match_span(lower, SECTION_PATTERNS["item7"])
    item7a_span = first_match_span(lower, SECTION_PATTERNS["item7a"])

    sections = {}

    if item1_span:
        start = item1_span[0]
        end = item1a_span[0] if item1a_span else (item7_span[0] if item7_span else min(len(text), start + 120000))
        sections["item1"] = text[start:end]

    if item7_span:
        start = item7_span[0]
        end = item7a_span[0] if item7a_span else min(len(text), start + 160000)
        sections["item7"] = text[start:end]

    if "item1" not in sections:
        sections["item1"] = text[:120000]

    if "item7" not in sections:
        mid = len(text) // 2
        sections["item7"] = text[mid:min(len(text), mid + 120000)]

    return sections

Cell 9 — Paragraph cleanup and splitting

Breaks the extracted filing text into paragraphs and removes obvious noise like very short fragments, table-heavy blocks, and SEC boilerplate. This gives the ranking step cleaner candidate passages.

In [ ]:
def is_junk_paragraph(p: str) -> bool:
    words = p.split()
    if len(words) < 25:
        return True

    digits = len(re.findall(r"\b\d[\d,.\-]*\b", p))
    if digits > 25 and len(words) < 120:
        return True

    junk_phrases = [
        "table of contents",
        "united states securities and exchange commission",
        "commission file number",
        "form 10-k",
        "page",
    ]
    p_lower = p.lower()
    if sum(phrase in p_lower for phrase in junk_phrases) >= 2:
        return True

    return False

def split_paragraphs(text: str):
    raw_parts = re.split(r"\n\s*\n+", text)
    paras = []
    for p in raw_parts:
        p = normalize_whitespace(p)
        if p and not is_junk_paragraph(p):
            paras.append(p)
    return paras

Cell 10 — Semantic similarity scoring

Scores each paragraph for relevance to private credit. It combines embedding-based semantic similarity with small bonuses for exact keyword hits and for appearing inside important sections like Item 1 or Item 7.

In [ ]:
def cosine_similarity_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a / np.linalg.norm(a, axis=1, keepdims=True)
    b = b / np.linalg.norm(b, axis=1, keepdims=True)
    return a @ b.T

def keyword_bonus(paragraph: str) -> float:
    p = paragraph.lower()
    groups_hit = 0
    for terms in KEYWORD_GROUPS.values():
        if any(term.lower() in p for term in terms):
            groups_hit += 1
    return 0.04 * groups_hit

def score_paragraphs(paragraphs, section_name, filing_label, embed_model, query_embeddings):
    para_embeddings = embed_model.encode(
        paragraphs,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )
    sims = cosine_similarity_matrix(para_embeddings, query_embeddings)

    rows = []
    for i, p in enumerate(paragraphs):
        max_sim = float(sims[i].max())
        hits_above_threshold = int((sims[i] > 0.42).sum())
        semantic_bonus = 0.02 * hits_above_threshold
        kw_bonus = keyword_bonus(p)
        section_bonus = 0.08 if section_name in {"item1", "item7"} else 0.0

        rows.append({
            "idx": i,
            "paragraph": p,
            "score": max_sim + semantic_bonus + kw_bonus + section_bonus,
            "section": section_name,
            "filing": filing_label,
            "word_count": len(p.split()),
        })
    return rows

Cell 11 — Paragraph selection with context

Selects the highest-scoring paragraphs while also keeping nearby context paragraphs before and after each hit. This helps preserve meaning so Gemini sees a coherent excerpt instead of isolated lines.

In [ ]:
def select_with_context(paragraphs, scored_rows, min_words=1200, max_words=1700):
    scored_rows = sorted(scored_rows, key=lambda x: x["score"], reverse=True)
    chosen = set()
    total_words = 0

    for row in scored_rows:
        center = row["idx"]
        window = [center - 1, center, center + 1]

        for j in window:
            if 0 <= j < len(paragraphs) and j not in chosen:
                wc = len(paragraphs[j].split())
                if total_words + wc > max_words:
                    continue
                chosen.add(j)
                total_words += wc

        if total_words >= min_words:
            break

    return sorted(chosen)

Cell 12 — Build excerpts from filings

Builds the final curated text bundle for the bank. It selects the strongest paragraphs from each filing, combines them, and enforces the target excerpt size of roughly 3,000–6,000 words.

In [ ]:
def build_excerpt_for_filing(text, filing_label, embed_model, query_embeddings, hard_cap_words=3000):
    sections = extract_sections(text)
    all_selected_paras = []
    debug_rows = []

    for sec_name in ["item1", "item7"]:
        paras = split_paragraphs(sections.get(sec_name, ""))
        if not paras:
            continue

        scored = score_paragraphs(paras, sec_name, filing_label, embed_model, query_embeddings)
        debug_rows.extend(scored)

        selected_idxs = select_with_context(
            paragraphs=paras,
            scored_rows=scored,
            min_words=1200,
            max_words=1700,
        )

        all_selected_paras.extend([
            f"[{filing_label} | {sec_name}] {paras[i]}" for i in selected_idxs
        ])

    excerpt = "\n\n".join(all_selected_paras)
    words = excerpt.split()
    if len(words) > hard_cap_words:
        excerpt = " ".join(words[:hard_cap_words])

    return excerpt, debug_rows

def build_bank_excerpt(filing_texts, embed_model, min_words=3000, max_words=6000):
    query_embeddings = embed_model.encode(
        QUERY_PHRASES,
        convert_to_numpy=True,
        normalize_embeddings=False,
        show_progress_bar=False,
    )

    all_chunks = []
    all_debug = []
    per_filing_cap = max_words // max(1, len(filing_texts))

    for filing_label, filing_text in filing_texts:
        excerpt, debug_rows = build_excerpt_for_filing(
            text=filing_text,
            filing_label=filing_label,
            embed_model=embed_model,
            query_embeddings=query_embeddings,
            hard_cap_words=max(2200, per_filing_cap),
        )
        all_chunks.append(excerpt)
        all_debug.extend(debug_rows)

    combined = "\n\n".join(chunk for chunk in all_chunks if chunk.strip())
    words = combined.split()

    if len(words) < min_words:
        global_rows = sorted(all_debug, key=lambda x: x["score"], reverse=True)
        existing = set()
        extra_parts = [combined] if combined.strip() else []
        current_words = len(words)

        for row in global_rows:
            para = f"[{row['filing']} | {row['section']}] {row['paragraph']}"
            if para in existing:
                continue
            wc = len(para.split())
            if current_words + wc > max_words:
                continue
            extra_parts.append(para)
            existing.add(para)
            current_words += wc
            if current_words >= min_words:
                break

        combined = "\n\n".join(part for part in extra_parts if part.strip())

    words = combined.split()
    if len(words) > max_words:
        combined = " ".join(words[:max_words])

    return combined

Cell 13 — Gemini client and report generation

Initializes the Gemini API connection and sends the curated excerpt to the model using the structured prompt. It also includes a helper to extract the final numeric score from Gemini’s response.

In [ ]:
def get_gemini_client():
    if not os.getenv("GEMINI_API_KEY"):
        raise EnvironmentError("GEMINI_API_KEY is not set.")
    return genai.Client()

def generate_report_with_gemini(client, bank_name, idrssd, excerpt, model_name=GEMINI_MODEL_NAME):
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(bank_name=bank_name)
    prompt = f"""
Bank: {bank_name}
IDRSSD: {idrssd}

Curated filing excerpts:
{excerpt}
"""
    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config={
            "system_instruction": system_prompt,
            "temperature": 0.2,
        },
    )
    return response.text.strip()

def parse_score(report_text):
    m = re.search(r"Score:\s*(\d{1,2})\s*/\s*10", report_text, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

Cell 14 — Load models and initialize client

Loads the local embedding model used for semantic ranking and initializes the Gemini client. This prepares both retrieval and generation components before the actual processing loop.

In [ ]:
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

print("Initializing Gemini client...")
client = get_gemini_client()

Cell 15 — Run the processing loop

Runs the full pipeline bank by bank: find filings, read them, extract sections, rank paragraphs, build the excerpt, send it to Gemini, and save both the curated text and the final report. This is the main execution step of the notebook.

In [ ]:
summary_rows = []
failures = []

for ticker in TICKERS_TO_RUN:
    if ticker not in BANKS:
        print(f"[SKIP] Unknown ticker: {ticker}")
        continue

    bank_meta = BANKS[ticker]
    bank_dir = ROOT_DIR / ticker

    print(f"\nProcessing {ticker} - {bank_meta['name']}")

    try:
        tenk_dirs = find_latest_two_10k_dirs(bank_dir)
        if not tenk_dirs:
            raise FileNotFoundError(f"No 10-K folders found for {ticker}")

        filing_texts = []
        used_folders = []

        for filing_dir in tenk_dirs:
            text = read_filing_text(filing_dir)
            filing_texts.append((filing_dir.name, text))
            used_folders.append(filing_dir.name)

        excerpt = build_bank_excerpt(
            filing_texts=filing_texts,
            embed_model=embed_model,
            min_words=3000,
            max_words=6000,
        )

        report = generate_report_with_gemini(
            client=client,
            bank_name=bank_meta["name"],
            idrssd=bank_meta["idrssd"],
            excerpt=excerpt,
        )

        excerpt_path = OUTPUT_DIR / "curated_excerpts" / f"{ticker}_excerpt.txt"
        report_path = OUTPUT_DIR / "reports" / f"{ticker}_private_credit_report.md"

        save_text(excerpt_path, excerpt)
        save_text(report_path, report)

        row = {
            "ticker": ticker,
            "bank_name": bank_meta["name"],
            "idrssd": bank_meta["idrssd"],
            "score": parse_score(report),
            "filings_used": used_folders,
            "excerpt_words": len(excerpt.split()),
            "excerpt_path": str(excerpt_path),
            "report_path": str(report_path),
        }
        summary_rows.append(row)

        print(f"  Filings used: {used_folders}")
        print(f"  Excerpt words: {row['excerpt_words']}")
        print(f"  Score: {row['score']}")
        print(f"  Report saved: {report_path}")

        time.sleep(1.5)

    except Exception as e:
        failures.append({"ticker": ticker, "error": str(e)})
        print(f"  ERROR: {e}")

Cell 16 — Save summary

In [ ]:
summary = {
    "completed": summary_rows,
    "failures": failures,
}
summary_path = OUTPUT_DIR / "summary.json"
save_text(summary_path, json.dumps(summary, indent=2))

print(f"\nDone. Summary saved to {summary_path}")